In [2]:
!pip install -q youtube-transcript-api langchain langchain-community langchain-huggingface langchain-chroma sentence-transformers chromadb tiktoken python-dotenv huggingface_hub

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\disha\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\transformers\\file_utils.py'
Consider using the `--user` option or check the permissions.



In [4]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

In [7]:
load_dotenv()
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [19]:
video_id = "J5_-l7WIO_w"  # e.g. the part after v= in the YouTube URL

try:
    api = YouTubeTranscriptApi()
    transcript_list = api.list(video_id)

    try:
        transcript_obj = transcript_list.find_transcript(["hi"])
    except NoTranscriptFound:
        available = list(transcript_list)
        transcript_obj = available[0].translate("en")

    fetched = transcript_obj.fetch()
    transcript = " ".join(chunk.text for chunk in fetched)
    print(transcript[:500])  # preview first 500 chars

except TranscriptsDisabled:
    transcript = ""
    print("No captions available for this video")
except NoTranscriptFound:
    transcript = ""
    print("No transcript found in any language for this video")

हाय गाइज़, माय नेम इज नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू करेंगे। अह पिछले वीडियो में हमने रैग पढ़ना शुरू किया था और हमने फोकस किया था रैग के अराउंड जो भी थ्योरी है उसको डिस्कस करने के ऊपर। मैंने आपको बताया था कि रैग क्या होता है? उसकी जरूरत क्यों होती है? मैंने वहां पर रैक को कंपेयर करके भी दिखाया था फाइन ट्यूनिंग जैसी टेक्निक के साथ। और आज का जो वीडियो है वह पिछले वीडियो का ही कंटिन्यूएशन है। जहां पर हम प्रैक्टिकली एक रैग बेस्ड सिस्


In [20]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 51


In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2207.09it/s]


In [14]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
    max_new_tokens=512,
    temperature=0.3
)

llm = ChatHuggingFace(llm=llm)

In [15]:
prompt = PromptTemplate(
    template="""You are a helpful assistant answering questions about a YouTube video.
Answer ONLY using the context below. If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

parallel_chain = RunnableParallel({
    "context": retriever | format_docs,
    "question": RunnablePassthrough()
})

parser = StrOutputParser()

main_chain = parallel_chain | prompt | llm | parser

In [16]:
def ask(question):
    answer = main_chain.invoke(question)
    print(f"Q: {question}\nA: {answer}\n")

ask("What is this video about?")

Q: What is this video about?
A: यह वीडियो एक प्रोजेक्ट के बारे में है जिसमें YouTube वीडियो का ट्रांसक्रिप्ट लोड करना और उसे मल्टीपल चंक्स में डिवाइड करना शामिल है। इसके बाद, एक रिट्रीवर को बनाना है जो उन चंक्स की एंबेडिंग्स जनरेट करेगा और उन्हें एक वेक्टर स्टोर में स्टोर करेगा।



In [ ]:
while True:
    question = input("Ask a question about the video (or type 'exit'): ")
    if question.lower() == "exit":
        break
    ask(question)